# NYC Building Permits ETL Pipeline
## Extract Phase

This pipeline extracts, transforms, and loads NYC building permit 
data from the NYC Department of Buildings into a Snowflake data 
warehouse. Data is sourced from the NYC Open Data API (Socrata) 
and encompasses permit issuance records across all five boroughs.

**Source:** NYC Department of Buildings — Socrata Open Data API
**Dataset:** DOB Permit Issuance
**Volume:** 50,000 most recent permit records

#### Import Libraries
Standard libraries for API requests, data handling, 
and environment variable management.

In [2]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully")

Libraries loaded successfully


#### Connect to Socrata API
Queries the NYC Open Data endpoint for building permit issuance 
records, retrieving the 50,000 most recent entries ordered by 
issuance date.

In [3]:
# Extract data from Socrata API
url = "https://data.cityofnewyork.us/resource/ipu4-2q9a.json"

params = {
    "$limit": 50000,
    "$offset": 0,
    "$order": "issuance_date DESC"
}

response = requests.get(url, params=params)
data = response.json()

print(f"Status code: {response.status_code}")
print(f"Records extracted: {len(data)}")

Status code: 200
Records extracted: 50000


#### Preview Raw Data
Initial inspection of the raw API response to assess 
structure, schema, and data types prior to transformation.

In [4]:
# Preview raw data structure
df_raw = pd.DataFrame(data)

print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nFirst record:")
df_raw.head(1)

Shape: (50000, 60)

Columns: ['borough', 'bin__', 'house__', 'street_name', 'job__', 'job_doc___', 'job_type', 'self_cert', 'block', 'lot', 'community_board', 'zip_code', 'bldg_type', 'permit_status', 'filing_status', 'permit_type', 'permit_sequence__', 'site_fill', 'filing_date', 'expiration_date', 'job_start_date', 'permittee_s_first_name', 'permittee_s_last_name', 'permittee_s_business_name', 'permittee_s_phone__', 'permittee_s_license_type', 'permittee_s_license__', 'site_safety_mgr_s_first_name', 'site_safety_mgr_s_last_name', 'site_safety_mgr_business_name', 'owner_s_business_type', 'non_profit', 'owner_s_business_name', 'owner_s_first_name', 'owner_s_last_name', 'owner_s_phone__', 'dobrundate', 'permit_si_no', 'gis_latitude', 'gis_longitude', 'gis_council_district', 'gis_census_tract', 'gis_nta_name', 'residential', 'work_type', 'special_district_1', 'permit_subtype', 'superintendent_first___last_name', 'superintendent_business_name', 'hic_license', 'permittee_s_other_title', 'a

,borough,bin__,house__,street_name,job__,job_doc___,job_type,self_cert,block,lot,...,permittee_s_other_title,act_as_superintendent,oil_gas,owner_s_house__,owner_s_house_street_name,city,state,owner_s_zip_code,issuance_date,special_district_2
0,BROOKLYN,3070624,25,STEWART AVE,320625070,01,NB,N,02994,00075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Select Relevant Columns
Schema narrowed from 60 to 20 columns, retaining fields 
relevant to permit activity, ownership, location, and 
job classification.

In [5]:
# Select relevant columns
columns_to_keep = [
    'borough', 'zip_code', 'block', 'lot',
    'job_type', 'permit_type', 'permit_status',
    'filing_date', 'issuance_date', 'expiration_date',
    'work_type', 'residential', 'bldg_type',
    'owner_s_business_name', 'owner_s_business_type',
    'permittee_s_business_name', 'permittee_s_license_type',
    'gis_latitude', 'gis_longitude', 'gis_nta_name'
]

df_selected = df_raw[columns_to_keep]

print(f"Columns before: {df_raw.shape[1]}")
print(f"Columns after: {df_selected.shape[1]}")
print(f"Rows: {df_selected.shape[0]}")

Columns before: 60
Columns after: 20
Rows: 50000


#### Assess Data Quality
Missing value analysis across all retained columns to 
inform transformation logic and document known data 
limitations.

In [6]:
# Assess data quality
missing = df_selected.isnull().sum()
missing_pct = (missing / len(df_selected) * 100).round(2)

quality_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

print(quality_report)

                           missing_count  missing_pct
borough                                0         0.00
zip_code                              20         0.04
block                                  9         0.02
lot                                    9         0.02
job_type                               0         0.00
permit_type                            0         0.00
permit_status                        200         0.40
filing_date                            0         0.00
issuance_date                      20979        41.96
expiration_date                    12115        24.23
work_type                          10232        20.46
residential                        28760        57.52
bldg_type                            234         0.47
owner_s_business_name               2208         4.42
owner_s_business_type               1614         3.23
permittee_s_business_name          16568        33.14
permittee_s_license_type           15047        30.09
gis_latitude                

Three categories of missing data:
1. Low missing (under 5%) — easy to handle:

zip_code, block, lot, permit_status, bldg_type, gis_latitude/longitude
We'll fill these or flag them as unknown

2. Medium missing (5–35%) — needs a decision:

owner_s_business_name (4.42%), permittee_s_business_name (33%), permittee_s_license_type (30%)
Makes sense — not every permit has a named business or licensed contractor

3. High missing (over 40%) — needs investigation:

residential (57.52%) — over half empty
issuance_date (41.96%) — nearly half have no issuance date
work_type (20.46%) — significant gap

- Many permits are filed but never formally issued, hence missing issuance_date.
- Residential field seems inconsistently filled by the city.

#### Save Raw Data Locally
Raw extracted data is saved as a CSV checkpoint prior to 
transformation, preserving the source data in its original 
state for auditability and reprocessing if needed.

In [11]:
# Save raw data as checkpoint
df_selected.to_csv('data/raw_permits.csv', index=False)
print(f"Raw data saved: {df_selected.shape[0]} rows, {df_selected.shape[1]} columns")

Raw data saved: 50000 rows, 20 columns
